# Módulo 5 · Práctica: azar controlado

Un LLM sortea cada token, arranca con pesos aleatorios y estima su pérdida con minibatches. En este notebook simulas el azar con NumPy, compruebas la ley de los grandes números y el teorema central del límite, y estimas parámetros por máxima verosimilitud.

**Cómo trabajar:** ejecuta en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás.

| Parte | Qué haces |
| --- | --- |
| 0 | Generador aleatorio reproducible |
| 1 | Muestrear una categórica (como un token) |
| 2 | Esperanza y varianza: teoría frente a simulación |
| 3 | Monte Carlo: estimar π |
| 4 | Ley de los grandes números |
| 5 | Teorema central del límite |
| 6 | Máxima verosimilitud: Bernoulli, gaussiana y categórica |
| 7 | Conexiones con LLM: inicialización y minibatches |
| 8 | Retos |

## Parte 0 · Un generador reproducible

Usa siempre `np.random.default_rng(semilla)`. Con la misma semilla obtienes los mismos números, igual que al fijar la semilla de un entrenamiento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, pi

rng = np.random.default_rng(0)
print(rng.random(3))          # uniformes en [0, 1)
print(rng.normal(0, 1, 3))    # gaussianas N(0, 1)
print(rng.integers(1, 7, 5))  # cinco tiradas de un dado

## Parte 1 · Muestrear una categórica

La salida de un LLM es una distribución categórica sobre el vocabulario. Para sortear un token: calcula la acumulada, genera u ~ U[0, 1) y elige el primer índice cuya acumulada supere u.

**Ejercicio 1.** Implementa `muestrear(p, u)`: devuelve el índice elegido para cada valor de `u` (un array). Pista: `np.cumsum` y `np.searchsorted(..., side='right')`.

In [ ]:
def muestrear(p, u):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
z = np.array([2., 1., 0., -1.])            # logits del documento (sección 5.1)
p = np.exp(z) / np.exp(z).sum()
assert list(muestrear(p, np.array([0.5, 0.9, 0.99]))) == [0, 2, 3], "Revisa la acumulada"
mues = muestrear(p, np.random.default_rng(1).random(100_000))
frec = np.bincount(mues, minlength=4) / len(mues)
assert np.allclose(frec, p, atol=0.01), "Las frecuencias deben parecerse a p"
print("Correcto. p =", p.round(4), " frecuencias =", frec.round(4))

## Parte 2 · Esperanza y varianza

E[X] = Σ x·p(x) y Var(X) = E[X²] − E[X]². Una simulación grande debe acercarse a la teoría.

**Ejercicio 2.** Para X con valores (0, 1, 2) y probabilidades (0,5; 0,3; 0,2), calcula `E` y `Var` exactas y sus estimaciones `E_mc`, `Var_mc` con 200.000 muestras.

In [ ]:
valores = np.array([0, 1, 2])
probs = np.array([0.5, 0.3, 0.2])
# TU CÓDIGO
E = Var = None
x = np.random.default_rng(2).choice(valores, size=200_000, p=probs)
E_mc = Var_mc = None

In [ ]:
# Comprobación
assert abs(E - 0.7) < 1e-12 and abs(Var - 0.61) < 1e-12, "E = 0,7 y Var = 0,61"
assert abs(E_mc - E) < 0.01 and abs(Var_mc - Var) < 0.01, "La simulación debería quedar cerca"
print(f"Correcto. E = {E:.3f} (MC {E_mc:.4f}), Var = {Var:.3f} (MC {Var_mc:.4f})")

## Parte 3 · Monte Carlo: estimar π

Lanza n puntos uniformes en el cuadrado [0, 1]². La fracción que cae dentro del cuarto de círculo x² + y² ≤ 1 estima π/4.

$$\hat\pi_n = 4\cdot\frac{\#\{x_i^2+y_i^2\le 1\}}{n}, \qquad \mathrm{error\ típico} = 4\sqrt{\frac{(\pi/4)(1-\pi/4)}{n}} \approx \frac{1{,}64}{\sqrt n}$$

**Ejercicio 3.** Implementa `estimar_pi(n, rng)`.

In [ ]:
def estimar_pi(n, rng):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
est = estimar_pi(1_000_000, np.random.default_rng(3))
assert abs(est - pi) < 0.01, "Con un millón de puntos el error debería ser < 0,01"
# El error típico debe escalar como 1/sqrt(n)
ns = [100, 1_000, 10_000, 100_000]
r = np.random.default_rng(4)
err = [np.std([estimar_pi(n, r) for _ in range(200)]) for n in ns]
teo = [4 * sqrt((pi/4) * (1 - pi/4) / n) for n in ns]
assert all(0.8 < e / t < 1.2 for e, t in zip(err, teo)), "El error no escala como 1/sqrt(n)"
print(f"Correcto. pi ≈ {est:.5f}")
for n, e, t in zip(ns, err, teo):
    print(f"n = {n:>7}: error típico simulado {e:.4f}, teórico {t:.4f}")
plt.loglog(ns, err, 'o-', label='simulado'); plt.loglog(ns, teo, '--', label='1,64/√n')
plt.xlabel('n'); plt.ylabel('error típico'); plt.legend(); plt.show()

## Parte 4 · Ley de los grandes números

La media de n tiradas de un dado converge a E[X] = 3,5. Lo vemos con la **media acumulada**: x̄ₖ = (x₁ + … + xₖ)/k.

**Ejercicio 4.** Implementa `media_acumulada(x)` sin bucles (pista: `np.cumsum` y `np.arange`).

In [ ]:
def media_acumulada(x):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert np.allclose(media_acumulada(np.array([2, 4, 6, 8])), [2, 3, 4, 5]), "Prueba con [2, 4, 6, 8]"
tiradas = np.random.default_rng(5).integers(1, 7, 100_000)
m = media_acumulada(tiradas)
assert abs(m[-1] - 3.5) < 0.03, "Con 100.000 tiradas la media debe estar muy cerca de 3,5"
print("Correcto. Media tras 10, 100, 1.000 y 100.000 tiradas:", m[[9, 99, 999, -1]].round(3))
plt.semilogx(np.arange(1, len(m) + 1), m); plt.axhline(3.5, ls='--', c='k')
plt.xlabel('número de tiradas'); plt.ylabel('media acumulada'); plt.show()

## Parte 5 · Teorema central del límite

Suma n variables independientes con media μ y varianza σ². La suma estandarizada

$$Z_n = \frac{S_n - n\mu}{\sigma\sqrt n}$$

se parece cada vez más a una N(0, 1), **sea cual sea** la distribución de partida. Usamos una exponencial (μ = σ = 1), muy asimétrica.

**Ejercicio 5.** Genera 50.000 valores de Z₃₀ (sumas de 30 exponenciales de media 1, estandarizadas) y calcula `frac`, la fracción con |Z| ≤ 1,96.

In [ ]:
r = np.random.default_rng(6)
# TU CÓDIGO
Z = None
frac = None

In [ ]:
# Comprobación
assert Z is not None and len(Z) == 50_000, "Z debe tener 50.000 valores"
assert abs(Z.mean()) < 0.02 and abs(Z.std() - 1) < 0.02, "Z debe tener media 0 y desviación 1"
assert abs(frac - 0.95) < 0.01, "Alrededor del 95 % debe caer en ±1,96"
print(f"Correcto. Fracción en ±1,96: {frac:.4f} (la gaussiana da {erf(1.96/sqrt(2)):.4f})")
t = np.linspace(-4, 4, 200)
plt.hist(Z, bins=80, density=True, alpha=0.6, label='Z₃₀ simulada')
plt.plot(t, np.exp(-t**2 / 2) / sqrt(2 * pi), 'k', label='N(0, 1)')
plt.legend(); plt.title('TCL con sumas de 30 exponenciales'); plt.show()

**Pregunta.** Repite con n = 1 y n = 5. ¿Cuánto se aleja la fracción de 0,95? ¿Por qué la cola derecha es más pesada?

## Parte 6 · Máxima verosimilitud

Elige los parámetros θ que hacen más probables los datos: maximiza ℓ(θ) = Σᵢ log p(xᵢ; θ). Es exactamente lo que hace el preentrenamiento de un LLM.

**Ejercicio 6 (Bernoulli).** 7 caras en 10 lanzamientos. Evalúa la log-verosimilitud `ll` en la rejilla `ps` y guarda en `p_mejor` el valor que la maximiza.

In [ ]:
k, n = 7, 10
ps = np.linspace(0.001, 0.999, 999)
# TU CÓDIGO
ll = None
p_mejor = None

In [ ]:
# Comprobación
assert abs(p_mejor - 0.7) < 1e-9, "El máximo está en k/n"
assert abs(ll.max() - (-6.1086)) < 1e-3, "ℓ(0,7) = −6,109"
print(f"Correcto. p_EMV = {p_mejor:.3f}, ℓ máx = {ll.max():.4f}")
plt.plot(ps, ll); plt.ylim(-20, -5); plt.axvline(0.7, ls='--', c='k')
plt.xlabel('p'); plt.ylabel('log-verosimilitud'); plt.show()

**Ejercicio 7 (gaussiana).** Implementa `emv_gauss(x)` que devuelva (μ̂, σ̂²) de máxima verosimilitud (σ̂² divide por n, no por n − 1).

In [ ]:
def emv_gauss(x):
    # TU CÓDIGO
    return None, None

In [ ]:
# Comprobación
mu, s2 = emv_gauss(np.array([2., 4, 4, 4, 5, 5, 7, 9]))
assert abs(mu - 5) < 1e-12 and abs(s2 - 4) < 1e-12, "Ejemplo del documento: μ = 5, σ² = 4"
datos = np.random.default_rng(7).normal(3.0, 2.0, 100_000)
mu, s2 = emv_gauss(datos)
assert abs(mu - 3) < 0.03 and abs(s2 - 4) < 0.08, "Debería recuperar μ = 3, σ² = 4"
# Sesgo: con n = 5, la media de σ̂² es (n-1)/n · σ² = 3,2
r = np.random.default_rng(8)
media_s2 = np.mean([emv_gauss(r.normal(3.0, 2.0, 5))[1] for _ in range(40_000)])
assert abs(media_s2 - 3.2) < 0.06, "El EMV de la varianza está sesgado: E = (n-1)/n · σ²"
print(f"Correcto. μ̂ = {mu:.3f}, σ̂² = {s2:.3f}; con n = 5, E[σ̂²] ≈ {media_s2:.3f} (teoría 3,2)")

**Ejercicio 8 (categórica y Laplace).** Un «LLM» de 4 tokens tiene probabilidades verdaderas `p_true`. Observas solo 20 tokens. Implementa `estimar(cuentas, alpha)` = (N + α)/(Σ N + α·K). Con α = 0 es el EMV; con α = 1, el suavizado de Laplace del módulo 1 (media posterior con prior de Dirichlet(1, …, 1)). Calcula la entropía cruzada de cada estimación sobre 10.000 tokens nuevos.

In [ ]:
p_true = np.array([0.6, 0.25, 0.1, 0.05])
r = np.random.default_rng(0)
train_tok = r.choice(4, size=20, p=p_true)
test_tok = r.choice(4, size=10_000, p=p_true)
cuentas = np.bincount(train_tok, minlength=4)

def estimar(cuentas, alpha):
    # TU CÓDIGO
    return None

def entropia_cruzada(q, tokens):
    # TU CÓDIGO: -media de log q[token]
    return None

In [ ]:
# Comprobación
assert np.allclose(estimar(np.array([3, 1, 0]), 1), [4/7, 2/7, 1/7]), "Ejemplo del documento: (4/7, 2/7, 1/7)"
print("Cuentas observadas:", cuentas)
q0, q1 = estimar(cuentas, 0), estimar(cuentas, 1)
h0, h1 = entropia_cruzada(q0, test_tok), entropia_cruzada(q1, test_tok)
assert np.isinf(h0), "Si un token no aparece en train, el EMV le da probabilidad 0 y la pérdida es infinita"
assert h1 < 1.5, "Con Laplace la pérdida debe ser finita y razonable"
print(f"Correcto. EMV: {q0.round(3)} → H = {h0};  Laplace: {q1.round(3)} → H = {h1:.4f} nats")
print(f"Entropía verdadera (cota inferior): {-(p_true*np.log(p_true)).sum():.4f} nats")

## Parte 7 · Conexiones con los LLM

**Inicialización.** Si y = Σⱼ wⱼ xⱼ con n entradas independientes, Var(y) = n·Var(w)·Var(x). Para que la señal no explote ni se apague, Var(w) = 1/n.

**Ejercicio 9.** Propaga un lote de vectores por 10 capas lineales de 512×512 con pesos N(0, σ²). Implementa `desv_final(sigma)`, la desviación típica de las activaciones tras la capa 10.

In [ ]:
d, capas = 512, 10

def desv_final(sigma, semilla=10):
    r = np.random.default_rng(semilla)
    h = r.normal(0, 1, (256, d))
    # TU CÓDIGO: aplica 10 veces h = h @ W con W ~ N(0, sigma²) de forma (d, d)
    return None

In [ ]:
# Comprobación
s_buena = desv_final(1 / np.sqrt(d))
s_mala = desv_final(1.0)
assert 0.8 < s_buena < 1.2, "Con σ = 1/√n la desviación debe mantenerse cerca de 1"
assert s_mala > 1e12, "Con σ = 1 debería crecer como √512¹⁰ ≈ 3,5·10¹³"
print(f"Correcto. σ = 1/√512 = {1/np.sqrt(d):.4f} → desv. final {s_buena:.3f};  σ = 1 → {s_mala:.3e}")

**Minibatches.** La pérdida de un minibatch es una media de B pérdidas por token: estima sin sesgo la pérdida total y su error típico es σ/√B (si los tokens fueran independientes).

**Ejercicio 10.** Simula pérdidas por token como exponenciales de media 2 nats (σ = 2). Para cada B en `Bs`, calcula `et[B]`: la desviación típica de 2.000 medias de minibatch.

In [ ]:
r = np.random.default_rng(11)
Bs = [16, 64, 256, 1024]
et = {}
# TU CÓDIGO

In [ ]:
# Comprobación
for B in Bs:
    assert abs(et[B] / (2 / np.sqrt(B)) - 1) < 0.08, f"Con B = {B} el error típico debería ser ≈ 2/√B"
print("Correcto. B, error típico simulado, 2/√B:")
for B in Bs:
    print(f"{B:>5}  {et[B]:.4f}  {2/np.sqrt(B):.4f}")

## Parte 8 · Retos

1. **π con menos varianza.** Estima π como 4·E[√(1 − U²)] con U uniforme. Compara su error típico con el método de los puntos para el mismo n.
2. **Temperatura y varianza.** Para los logits z = (2, 1, 0, −1), calcula la entropía de softmax(z/T) para T ∈ {0,5; 1; 2} y comprueba con muestreo.
3. **He frente a Xavier.** Añade ReLU tras cada capa del Ejercicio 9. Verás que con σ = 1/√n la señal se apaga; con σ = √(2/n) (He, 2015) se mantiene. ¿Por qué el factor 2?
4. **Intervalos para benchmarks.** Un modelo acierta 820 de 1.000 preguntas. Construye su intervalo del 95 % con el TCL y compruébalo con *bootstrap* (remuestrear las 1.000 respuestas con reemplazo).

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def muestrear(p, u):
#     acum = np.cumsum(p)
#     idx = np.searchsorted(acum, u, side='right')
#     return np.minimum(idx, len(p) - 1)
# Ejercicio 2
# E = (valores * probs).sum()
# Var = (valores**2 * probs).sum() - E**2
# E_mc, Var_mc = x.mean(), x.var()
# Ejercicio 3
# def estimar_pi(n, rng):
#     x, y = rng.random(n), rng.random(n)
#     return 4 * np.mean(x**2 + y**2 <= 1)
# Ejercicio 4
# def media_acumulada(x):
#     return np.cumsum(x) / np.arange(1, len(x) + 1)
# Ejercicio 5
# n = 30
# S = r.exponential(1.0, size=(50_000, n)).sum(axis=1)
# Z = (S - n * 1.0) / (1.0 * np.sqrt(n))
# frac = np.mean(np.abs(Z) <= 1.96)
# Ejercicio 6
# ll = k * np.log(ps) + (n - k) * np.log(1 - ps)
# p_mejor = ps[np.argmax(ll)]
# Ejercicio 7
# def emv_gauss(x):
#     mu = x.mean()
#     return mu, np.mean((x - mu)**2)
# Ejercicio 8
# def estimar(cuentas, alpha):
#     return (cuentas + alpha) / (cuentas.sum() + alpha * len(cuentas))
# def entropia_cruzada(q, tokens):
#     with np.errstate(divide='ignore'):
#         return -np.mean(np.log(q[tokens]))
# Ejercicio 9
# def desv_final(sigma, semilla=10):
#     for _ in range(capas):
#         h = h @ r.normal(0, sigma, (d, d))
#     return h.std()
# Ejercicio 10
# for B in Bs:
#     medias = r.exponential(2.0, size=(2000, B)).mean(axis=1)
#     et[B] = medias.std()